<font color='green'>***Installation and Libraries Import***</font>
---
--- 

In [1]:
# Source - https://stackoverflow.com/a
# Posted by Nielsou Akbrg
# Retrieved 2026-01-27, License - CC BY-SA 4.0



# %pip install flwr
# %pip install ray 
# %pip install --upgrade pip
%pip install torch torchvision matplotlib
# %pip install async-timeout
# %pip install async-numpy
# %pip install pandas
# %pip install datasets
# %pip install scikit-learn



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================
# TensorFlow / TensorBoard Cleanup
# ============================

import os
import flwr as fl
import ray
import torch

libraries_to_uninstall = [
    "tb-nightly==2.18.0a20240701",
    "tensorboard==2.16.2",
    "tensorboard-data-server==0.7.2",
    "tensorboard-plugin-wit==1.8.1",
    "tensorflow==2.16.2",
    "tensorflow-io-gcs-filesystem==0.37.0",
    "termcolor==2.4.0",
    "terminado==0.18.1",
    "tf-estimator-nightly==2.8.0.dev2021122109",
    "tf_keras-nightly==2.18.0.dev2024070109",
    "tf-nightly==2.18.0.dev20240626",
]

for library in libraries_to_uninstall:
    os.system(f"pip uninstall -y {library}")

print("Cleanup complete.")
print("FLWR version:", fl.__version__)
print("Ray version:", ray.__version__)
print("Torch version:", torch.__version__)

Cleanup complete.
FLWR version: 1.5.0
Ray version: 2.31.0
Torch version: 2.8.0+cu128


In [3]:
# ============================
# Core Imports & Environment Setup
# ============================

from collections import OrderedDict, Counter
from typing import  List, Optional, Tuple

import os
import sys
import math
import time
import random
import copy
import csv
import pickle
import importlib
import glob

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision.transforms as transforms
from torch.utils.data import (
    Dataset,
    DataLoader,
    random_split,
    Subset,
    ConcatDataset,
    WeightedRandomSampler,
)

import flwr as fl
from flwr.common import Metrics

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

import ray

import llm
from llm import llm_mid_training_update

import benchmark
from benchmark import BenchmarkLogger

from pprint import pprint

print("Python executable:", sys.executable)
print("FLWR version:", fl.__version__)
print("Ray version:", ray.__version__)
print("Torch version:", torch.__version__)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

Python executable: /home/zeus/miniconda3/envs/cloudspace/bin/python
FLWR version: 1.5.0
Ray version: 2.31.0
Torch version: 2.8.0+cu128
DEVICE: cuda:0


<font color='Brown'>***Constants***</font>
---
--- 

In [4]:
# ------------------------
# Configuration
# ------------------------
import json

with open("config.json", "r") as f:
    CFG = json.load(f)

NUM_CLIENTS = CFG["NUM_CLIENTS"]
ROUNDS = CFG["ROUNDS"]
BATCH_SIZE = CFG["BATCH_SIZE"]
LEARNING_RATE = CFG["LEARNING_RATE"]
EPOCHS = CFG["EPOCHS"]
DATA_GROUPS = CFG["DATA_GROUPS"]
BATCH_ROUND = CFG["BATCH_ROUND"]
NUM_ATCKS = CFG["NUM_ATCKS"]
FL = CFG["FL"]
MODE = CFG["MODE"]
G = CFG["G"]
LLM = CFG["LLM"]
LLM_TRIGGER_ROUND = CFG["LLM_TRIGGER_ROUND"]
DATA_DIR = CFG["DATA_DIR"]

SIZE_ROUND = BATCH_ROUND * BATCH_SIZE * NUM_CLIENTS

PATH = CFG["PATH_TEMPLATE"].format(
    MODE=MODE,
    FL=FL,
    NUM_CLIENTS=NUM_CLIENTS,
    NUM_ATCKS=NUM_ATCKS,
    ROUNDS=ROUNDS,
    EPOCHS=EPOCHS,
    LEARNING_RATE=LEARNING_RATE,
    DATA_GROUPS=DATA_GROUPS,
    LLM=LLM,
)

os.makedirs(PATH, exist_ok=True)

# ------------------------
# Per-experiment state
# ------------------------
round_accuracy = {}
round_loss = {}

<font color='Light Blue'>***Dataset Preparations***</font>
---
--- 

In [5]:
# Dataset container
TrafficData = {}
TrafficData['Dataset']={}

print("CWD:", os.getcwd())

# Dataset variants by normal-traffic percentage
sets_names = ['30','100','70','50','120']
for DATA_NUM in sets_names:
    TrafficData['Dataset'][DATA_NUM] = pd.read_csv(
        os.path.join(
            DATA_DIR,
            f"2_Dataset_{NUM_ATCKS}_Attack_{DATA_NUM}_normal.csv"
        ),
        low_memory=False,
        quoting=csv.QUOTE_NONE,
        on_bad_lines="skip",
    )
    print(DATA_NUM, TrafficData['Dataset'][DATA_NUM].shape)

# Shuffle each dataset
for DATA_NUM in TrafficData['Dataset']:
    TrafficData['Dataset'][DATA_NUM]=TrafficData['Dataset'][DATA_NUM].sample(frac=1, random_state=42).reset_index(drop=True)

CWD: /teamspace/studios/this_studio
30 (184320, 99)
100 (184320, 99)
70 (184320, 99)
50 (184320, 99)
120 (120000, 99)


In [6]:
# Split datasets into per-round groups
TrafficData['Split'] = {}
sets_training =  ['30','100','70','50']

# Create DATA_GROUPS splits for each training dataset
for DATA_NUM in sets_training:
    TrafficData['Split'][DATA_NUM] = np.array_split(TrafficData['Dataset'][DATA_NUM],DATA_GROUPS)

# Combine corresponding splits across datasets
TrafficData['Combined'] = pd.concat([TrafficData['Split']['30'][0], TrafficData['Split']['100'][0], TrafficData['Split']['70'][0], TrafficData['Split']['50'][0]]).reset_index(drop=True)

# Append remaining split groups
for GROUP in range(1, DATA_GROUPS):
    TrafficData['Combined'] = pd.concat([TrafficData['Combined'], TrafficData['Split']['30'][GROUP], TrafficData['Split']['100'][GROUP], TrafficData['Split']['70'][GROUP], TrafficData['Split']['50'][GROUP]]).reset_index(drop=True)
print(TrafficData['Combined'].shape)            

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


(737280, 99)


In [7]:
# Create training feature/label split
TrafficData['Train'] = {}
TrafficData['Train']['X'] = TrafficData['Combined'].iloc[:, 0:-1]
TrafficData['Train']['y'] = TrafficData['Combined'].iloc[:, -1]
print(TrafficData['Train']['X'].shape)
print(TrafficData['Train']['y'].shape)

# Create test feature/label split
TrafficData['Test'] = {}
TrafficData['Test']['X']=TrafficData['Dataset']['120'].iloc[:, 0:-1]
TrafficData['Test']['y']=TrafficData['Dataset']['120'].iloc[:, -1]
print(TrafficData['Test']['X'].shape)
print(TrafficData['Test']['y'].shape)

(737280, 98)
(737280,)
(120000, 98)
(120000,)


In [8]:
# Select scaler based on model type
scaler = scaler = StandardScaler() if MODE == "TT" else MinMaxScaler()
model = scaler.fit(TrafficData['Train']['X'])

# Apply scaling to train and test features
TrafficData['Train']['X'] = model.transform(TrafficData['Train']['X'])
TrafficData['Test']['X'] = model.transform(TrafficData['Test']['X'])

# Convert training data to numpy arrays
TrafficData['Train']['X'], TrafficData['Train']['y']= np.array(TrafficData['Train']['X']), np.array(TrafficData['Train']['y'])
print(type(TrafficData['Train']['X']),type(TrafficData['Train']['y']))
print(TrafficData['Train']['X'].shape,TrafficData['Train']['y'].shape)

# Convert test data to numpy arrays
TrafficData['Test']['X'], TrafficData['Test']['y']= np.array(TrafficData['Test']['X']), np.array(TrafficData['Test']['y'])
print(type(TrafficData['Test']['X']),type(TrafficData['Test']['y']))
print(TrafficData['Test']['X'].shape,TrafficData['Test']['y'].shape)

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(737280, 98) (737280,)
<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(120000, 98) (120000,)


In [9]:
# Initialize per-round data containers
TrafficData['ROUNDS']={}
for ROUND in range(1, ROUNDS+1):
    TrafficData['ROUNDS'][ROUND]={}

# Slice training data into round-sized chunks
SIZE_Demo = SIZE_ROUND
for ROUND in range(1,ROUNDS+1):
    if ROUND == 1:
        TrafficData['ROUNDS'][ROUND]['X']= TrafficData['Train']['X'][:SIZE_Demo]
        TrafficData['ROUNDS'][ROUND]['y']= TrafficData['Train']['y'][:SIZE_Demo]
    else:
        print((SIZE_Demo - SIZE_ROUND),SIZE_Demo)
        TrafficData['ROUNDS'][ROUND]['X']= TrafficData['Train']['X'][(SIZE_Demo - SIZE_ROUND):SIZE_Demo]
        TrafficData['ROUNDS'][ROUND]['y']= TrafficData['Train']['y'][(SIZE_Demo - SIZE_ROUND):SIZE_Demo]
    SIZE_Demo = SIZE_Demo + SIZE_ROUND

# Log per-round dataset shapes
for ROUND in TrafficData['ROUNDS']:
    print("ROUND: ", ROUND, TrafficData['ROUNDS'][ROUND]['X'].shape, TrafficData['ROUNDS'][ROUND]['y'].shape)
print(SIZE_Demo, SIZE_ROUND)

18432 36864
36864 55296
55296 73728
73728 92160
92160 110592
110592 129024
129024 147456
ROUND:  1 (18432, 98) (18432,)
ROUND:  2 (18432, 98) (18432,)
ROUND:  3 (18432, 98) (18432,)
ROUND:  4 (18432, 98) (18432,)
ROUND:  5 (18432, 98) (18432,)
ROUND:  6 (18432, 98) (18432,)
ROUND:  7 (18432, 98) (18432,)
ROUND:  8 (18432, 98) (18432,)
165888 18432


In [10]:
class ClassifierDataset(Dataset):
    """
    PyTorch Dataset wrapper for feature and label tensors.
    """

    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data


    def __getitem__(self, index):
        """
        Returns a single (feature, label) pair at the given index.
        """
        return self.X_data[index], self.y_data[index]


    def __len__ (self):

        """
        Returns the total number of samples in the dataset.
        """   
        return len(self.X_data)

In [11]:
# Build per-round training datasets
TrafficData['trainsets']={}
for ROUND in range(1, ROUNDS+1):
    TrafficData['trainsets'][ROUND]= ClassifierDataset(torch.from_numpy(TrafficData['ROUNDS'][ROUND]['X']).float(), torch.from_numpy(TrafficData['ROUNDS'][ROUND]['y']).long())

# Build test dataset
TrafficData['testset'] = ClassifierDataset(torch.from_numpy(TrafficData['Test']['X']).float(), torch.from_numpy(TrafficData['Test']['y']).long())

In [12]:
def load_train(numberofclients, ROUND):    
    """
    Create balanced DataLoaders for each client for a given training round.

    The dataset is split as evenly as possible across clients. If the total
    number of samples is not divisible by the number of clients, the first
    few clients receive one additional sample (remainder-based distribution).
    """

    # Get dataset size for this round
    dataset_len = len(TrafficData['trainsets'][ROUND])
    
    # Return an empty list of loaders to avoid indexing errors downstream
    if dataset_len == 0:
        return [DataLoader(Subset(TrafficData['trainsets'][ROUND], []), batch_size=BATCH_SIZE, shuffle=False) for _ in range(numberofclients)]
    
    # Compute balanced split sizes
    num_portions = int(numberofclients)
    base_portion = dataset_len // num_portions
    remainder = dataset_len % num_portions

    # Build index ranges per client
    portion_indices = []
    start_idx = 0
    for i in range(num_portions):
        # First 'remainder' clients receive one extra sample
        sz = base_portion + (1 if i < remainder else 0)
        end_idx = start_idx + sz
        if sz > 0:
            portion_indices.append(list(range(start_idx, end_idx)))
        else:
            portion_indices.append([])
        start_idx = end_idx
    
    # Create per-client datasets
    portion_datasets = [Subset(TrafficData['trainsets'][ROUND], indices) for indices in portion_indices]

    # Wrap each dataset in a DataLoader
    portion_loaders = [DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False) for dataset in portion_datasets]            
    return portion_loaders

def load_test(numberofclients):    
    """
    Create a DataLoader for the shared test dataset.

    A single test DataLoader is used and shared across all clients.
    """
    testloader = DataLoader(TrafficData['testset'], batch_size=BATCH_SIZE, shuffle=False)
    return testloader

In [13]:
# Build training and test dataloaders
Dataloaders = {}
for ROUND in range(1, ROUNDS+1):
    Dataloaders[ROUND] = load_train(NUM_CLIENTS, ROUND)
Dataloaders['Test'] = load_test(NUM_CLIENTS)

In [14]:
# Diagnostic check for per-client dataset sizes
for ROUND in range(1, min(6, ROUNDS+1)):
    loaders = Dataloaders.get(ROUND, None)
    if loaders is None:
        print(f"Dataloaders for Round {ROUND} not found (run the rebuild cell).")
        continue
    
    sizes = [len(loader.dataset) for loader in loaders]
    print(f"Round {ROUND}: per-client sizes (first 12): {sizes[:12]}")
    print(f"  Total samples this round: {sum(sizes)}\n")

Round 1: per-client sizes (first 12): [384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384]
  Total samples this round: 18432

Round 2: per-client sizes (first 12): [384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384]
  Total samples this round: 18432

Round 3: per-client sizes (first 12): [384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384]
  Total samples this round: 18432

Round 4: per-client sizes (first 12): [384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384]
  Total samples this round: 18432

Round 5: per-client sizes (first 12): [384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384]
  Total samples this round: 18432



In [15]:
# Verify test batch sizes
for i, batch in enumerate(Dataloaders['Test']):
    batch_size = len(batch[0])  
    print(f"Batch {i+1} size: {batch_size}")
    if batch_size != 64:
        print(f"Batch {i+1} does not contain 64 records.")
        break

Batch 1 size: 64
Batch 2 size: 64
Batch 3 size: 64
Batch 4 size: 64
Batch 5 size: 64
Batch 6 size: 64
Batch 7 size: 64
Batch 8 size: 64
Batch 9 size: 64
Batch 10 size: 64
Batch 11 size: 64
Batch 12 size: 64
Batch 13 size: 64
Batch 14 size: 64
Batch 15 size: 64
Batch 16 size: 64
Batch 17 size: 64
Batch 18 size: 64
Batch 19 size: 64
Batch 20 size: 64
Batch 21 size: 64
Batch 22 size: 64
Batch 23 size: 64
Batch 24 size: 64
Batch 25 size: 64
Batch 26 size: 64
Batch 27 size: 64
Batch 28 size: 64
Batch 29 size: 64
Batch 30 size: 64
Batch 31 size: 64
Batch 32 size: 64
Batch 33 size: 64
Batch 34 size: 64
Batch 35 size: 64
Batch 36 size: 64
Batch 37 size: 64
Batch 38 size: 64
Batch 39 size: 64
Batch 40 size: 64
Batch 41 size: 64
Batch 42 size: 64
Batch 43 size: 64
Batch 44 size: 64
Batch 45 size: 64
Batch 46 size: 64
Batch 47 size: 64
Batch 48 size: 64
Batch 49 size: 64
Batch 50 size: 64
Batch 51 size: 64
Batch 52 size: 64
Batch 53 size: 64
Batch 54 size: 64
Batch 55 size: 64
Batch 56 size: 64
B

In [16]:
# Build LLM-compatible result snapshots for centralized and federated training

def build_llm_results_snapshot(acc_list, loss_list):
    """
    Build the minimal structure lam.py expects.
    """
    return {
        "Accuracy": acc_list.copy(),
        "Loss": loss_list.copy(),
        "Precision": [],
        "Recall": [],
        "F1_Score": [],
    }

def build_llm_fl_results_snapshot(global_acc, client_accs):
    """
    Build the exact structure lam.py expects for FL.
    
    global_acc: List[float]  (per-round global accuracy)
    client_accs: List[List[float]]  (per-round per-client accuracy)
    """

    mean_acc = [float(np.mean(c)) for c in client_accs]
    std_acc  = [float(np.std(c)) for c in client_accs]
    min_acc  = [float(np.min(c)) for c in client_accs]
    max_acc  = [float(np.max(c)) for c in client_accs]

    return {
        "global_accuracy": global_acc,
        "mean_client_accuracy": mean_acc,
        "std_client_accuracy": std_acc,
        "min_client_accuracy": min_acc,
        "max_client_accuracy": max_acc,
    }


In [17]:
# Verify batch sizes for a specific round and client
for i, batch in enumerate(Dataloaders[5][0]):
    batch_size = len(batch[0])  
    print(f"Batch {i+1} size: {batch_size}")
    if batch_size != 64:
        print(f"Batch {i+1} does not contain 64 records.")
        break

Batch 1 size: 64
Batch 2 size: 64
Batch 3 size: 64
Batch 4 size: 64
Batch 5 size: 64
Batch 6 size: 64


In [18]:
# Analyze class-0 percentage distribution across clients and clusters
for CLUSTER in range (1, 9):
    DEVICE_PERCENTAGE = []

    # Compute class-0 percentage per client
    for DEVICE__ in range(0,NUM_CLIENTS):
        for i, batch in enumerate(Dataloaders[CLUSTER][DEVICE__]):
            _, labels = batch
            class_counts = Counter(labels.numpy())
            total_records = sum(class_counts.values())
            class_0_count = class_counts.get(0, 0)
            percentage_class_0 = (class_0_count / total_records) * 100
            DEVICE_PERCENTAGE.append(percentage_class_0)
   
    # Average percentages in fixed-size chunks
    chunk_size = 6
    averages = [sum(DEVICE_PERCENTAGE[i:i + chunk_size]) / chunk_size for i in range(0, len(DEVICE_PERCENTAGE), chunk_size)]

    # Further aggregate averages across clusters    
    chunk_size_4 = 4
    averages = [sum(averages[i:i + chunk_size_4]) / chunk_size_4 for i in range(0, len(averages), chunk_size_4)]

    print(averages)

[61.653645833333336, 62.5, 60.15625, 62.3046875, 60.7421875, 60.87239583333333, 62.76041666666667, 63.020833333333336, 66.796875, 64.12760416666667, 62.95572916666667, 64.32291666666667]
[63.0859375, 60.872395833333336, 62.630208333333336, 62.5, 62.434895833333336, 63.151041666666664, 61.71875, 62.3046875, 62.044270833333336, 61.979166666666664, 60.546875, 62.5]
[60.872395833333336, 60.15625, 60.026041666666664, 63.411458333333336, 61.1328125, 61.979166666666664, 63.671875, 64.51822916666667, 63.4765625, 63.151041666666664, 64.97395833333333, 65.8203125]
[63.346354166666664, 63.4765625, 64.77864583333333, 61.588541666666664, 65.10416666666666, 62.56510416666667, 62.04427083333333, 63.216145833333336, 61.263020833333336, 61.00260416666667, 61.393229166666664, 61.39322916666667]
[63.020833333333336, 61.19791666666667, 62.369791666666664, 60.807291666666664, 58.59375, 60.677083333333336, 59.700520833333336, 62.5, 63.34635416666667, 62.6953125, 63.606770833333336, 63.606770833333336]
[63.3

In [19]:
# del TrafficData

<font color='Red'>***Neural Network***</font>
---
--- 

In [20]:
# Define DNN model architecture
if MODE == 'DNN':
    class Net(nn.Module):
        """
        Deep neural network for centralized or federated training.
        """
        def __init__(self):
            super(Net, self).__init__()        
            self.layer_1 = nn.Linear(98, 64)
            self.layer_2 = nn.Linear(64, 32)
            self.layer_out = nn.Linear(32, NUM_ATCKS+1) 
            self.relu = nn.ReLU()
            
        def forward(self, x):
            x = self.layer_1(x)
            x = self.relu(x)
            x = self.layer_2(x)
            x = self.relu(x)
            x = self.layer_out(x)        
            return x

<font color='Yellow'>***Tab Transformer***</font>
---
--- 

In [21]:
if MODE == 'TT':
    class TabTransformer(nn.Module):
        """
        TabTransformer model for tabular data using self-attention over features.
        """
        
        def __init__(
            self,
            num_features,
            emb_dim=32,
            num_layers=3,
            num_heads=4,
            mlp_dim=64,
            num_classes=NUM_ATCKS + 1,
        ):
            super().__init__()
            self.num_features = num_features
            self.emb_dim = emb_dim

            # Project scalar features into embedding space
            self.feature_projection = nn.Linear(1, emb_dim)

            # Learnable feature identifier embeddings
            self.feature_id_emb = nn.Parameter(torch.randn(num_features, emb_dim))

            # Transformer encoder over feature embeddings
            encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=num_heads, dim_feedforward=mlp_dim, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

            # Classification head
            self.output_head = nn.Sequential(
                nn.Linear(emb_dim, mlp_dim),
                nn.ReLU(),
                nn.Linear(mlp_dim, num_classes)
            )

        def forward(self, x):
            x = x.unsqueeze(-1)
            emb = self.feature_projection(x) 
            emb = emb + self.feature_id_emb.unsqueeze(0) 
            z = self.transformer(emb) 
            pooled = z.mean(dim=1) 
            logits = self.output_head(pooled) 
            return logits




<font color='Purple'>***Model Selection***</font>
---
--- 

In [22]:
# Model factory based on selected architecture
num_features = TrafficData["Train"]["X"].shape[1]
def build_model():
    """
    Instantiate the model specified by MODE.
    """
    if MODE == "DNN":
        return Net()
    elif MODE == "TT":
        return TabTransformer(num_features)
    else:
        raise ValueError(f"Unknown MODE={MODE}")

In [23]:
# Create and save an initial global model checkpoint for federated learning
init_ckpt = f"0_Input_Random_Model_{MODE}.pth"

model = build_model()
model.eval()

torch.save(model.state_dict(), init_ckpt)
print(f" Initial model saved: {init_ckpt}")


 Initial model saved: 0_Input_Random_Model_TT.pth


<font color='Orange'>***Train & Test Functions***</font>
---
--- 

In [24]:
def train(model, trainloader, epochs: int, lr:float,verbose=True):
    """
    Train a model for a given number of epochs on a single DataLoader.

    Returns per-epoch accuracy and loss along with prediction and label traces
    for downstream logging and LLM analysis.
    """

    # Loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()

    # Containers for logging
    prediction_matrix = []
    actual_matrix = []
    acc_matrix = []
    loss_matrix = []

    # Safeguard against empty datasets
    try:
        dataset_len = len(trainloader.dataset)
    except Exception:
        dataset_len = 0

    if dataset_len == 0:
        if verbose:
            print("Warning: trainloader has 0 samples, skipping training.")
        return prediction_matrix, actual_matrix, acc_matrix, loss_matrix

    # Training loop
    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0

        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Accumulate loss and accuracy
            epoch_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()

            predictions = torch.max(outputs.data, 1)[1]
            prediction_matrix.append(predictions.tolist())
            actual_matrix.append(labels.tolist())

        # Compute epoch metrics safely
        if total == 0:
            epoch_loss_val = 0.0
            epoch_acc = 0.0
        else:
            epoch_loss_val = epoch_loss / total
            epoch_acc = correct / total

        loss_matrix.append(epoch_loss_val)
        acc_matrix.append(epoch_acc)
        
        if verbose:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss_val:.6f} | Acc: {epoch_acc:.4f}")
    return prediction_matrix, actual_matrix, acc_matrix, loss_matrix 


def test(model, testloader):
    """
    Evaluate a model on the test dataset.

    Returns average loss, accuracy, and prediction/label traces for logging
    and downstream LLM analysis.
    """
    
    # Loss function and evaluation mode
    criterion = nn.CrossEntropyLoss()
    model.eval()
    model.to(DEVICE) 

    # Containers for metrics and traces
    correct, total, loss_sum = 0, 0, 0.0
    prediction_matrix = []
    actual_matrix = []
    acc_matrix = []
    loss_matrix = []

    # Evaluation loop (no gradient tracking)
    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(DEVICE)   #MOVE DATA
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss_sum += criterion(outputs, labels).item() * labels.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            prediction_matrix.append(predicted.cpu().tolist())
            actual_matrix.append(labels.cpu().tolist())

    # Compute final metrics safely  
    avg_loss = loss_sum / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0

    acc_matrix.append(accuracy)
    loss_matrix.append(avg_loss)

    print(f"Evaluation: loss={avg_loss:.6f}, acc={accuracy:.4f}")
    
    return avg_loss, accuracy, prediction_matrix, actual_matrix, acc_matrix, loss_matrix


***Centralized DNN/TabTransformer Rounds***
---

In [25]:
# Centralized Training Loop DNN
importlib.reload(benchmark)
importlib.reload(llm)

if not FL:
    def train_one_round(model, trainloader, epochs,lr):
        """
        Run multiple epochs of training on a single round dataset.
        """

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        model.train()

        for epoch in range(epochs):
            correct, total, running_loss = 0, 0, 0.0

            for Xbatch, ybatch in trainloader:
                Xbatch, ybatch = Xbatch.to(DEVICE), ybatch.to(DEVICE)

                optimizer.zero_grad()
                outputs = model(Xbatch)
                loss = criterion(outputs, ybatch)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                total += ybatch.size(0)
                correct += (torch.max(outputs, 1)[1] == ybatch).sum().item()

            print(f"  Epoch {epoch+1}/{epochs} | "
                f"Loss: {running_loss/len(trainloader):.6f} | "
                f"Acc: {correct/total:.4f}")


    def test_model(model, testloader):
        """
        Run a single evaluation pass without saving predictions.
        """

        criterion = nn.CrossEntropyLoss()
        model.eval()

        correct, total, running_loss = 0, 0, 0.0

        with torch.no_grad():
            for Xbatch, ybatch in testloader:
                Xbatch, ybatch = Xbatch.to(DEVICE), ybatch.to(DEVICE)
                outputs = model(Xbatch)
                loss = criterion(outputs, ybatch).item()

                running_loss += loss
                total += ybatch.size(0)
                correct += (torch.max(outputs, 1)[1] == ybatch).sum().item()

        avg_loss = running_loss / len(testloader)
        accuracy = correct / total

        print(f"  Test → Loss: {avg_loss:.6f}, Acc: {accuracy:.4f}")
        return avg_loss, accuracy


    # Centralized training loop
    experiment_start_time = time.time()
    llm_trigger_time = None

    bench = BenchmarkLogger(
        output_path=PATH,
        mode="centralized",
        model_type=MODE
    )

    os.makedirs(PATH, exist_ok=True)
    print(" Starting centralized Round-by-Round training...\n")

    model = build_model().to(DEVICE)

    for Round in range(1, ROUNDS + 1):
        bench.start_round(Round)

        print("========================")
        print(f"   ROUND {Round} START")
        print("========================")

        # Build round dataset from client partitions
        client_datasets = [
            loader.dataset for loader in Dataloaders[Round]
            if hasattr(loader, "dataset") and len(loader.dataset) > 0
        ]

        round_dataset = ConcatDataset(client_datasets)
        trainloader = DataLoader(
            round_dataset, 
            batch_size=BATCH_SIZE, 
            shuffle=True
        )

        print(f"  Train samples this round: {len(round_dataset)}")

        # Train
        t0 = time.time()
        train_one_round(
            model, 
            trainloader, 
            EPOCHS,LEARNING_RATE
        )
        train_time = time.time() - t0


        # Evaluate (optional)
        print(f"  Evaluating round {Round} model...")
        t0 = time.time()
        avg_loss, avg_acc = test_model(model, Dataloaders["Test"])
        eval_time = time.time() - t0

        # Store metrics
        round_accuracy[Round] = [avg_acc]
        round_loss[Round] = [avg_loss]
        samples_this_round = len(round_dataset)

        bench.log_memory()

        bench.end_round(
            train_time=train_time,
            eval_time=eval_time,
            accuracy=avg_acc,
            loss=avg_loss,
            samples_this_round=samples_this_round,
            learning_rate=LEARNING_RATE,
            epochs=EPOCHS,
        )


        # Save model
        save_path = f"{PATH}/GlobalModel_{Round}.pth"
        torch.save(model.state_dict(), save_path)
        print(f" Saved: {save_path}")

        print("========================")
        print(f"   ROUND {Round} DONE")
        print("========================\n")

        # Trigger LLM mid-training update
        if Round == LLM_TRIGGER_ROUND and LLM==True:
            llm_trigger_time = time.time()


            print("\nTriggering LLM mid-training update...")

            results_snapshot = build_llm_results_snapshot(
                acc_list=round_accuracy.get(Round, []),
                loss_list=round_loss.get(Round, []),

            )

            new_config = llm_mid_training_update(
                model_path=save_path,
                config_path="config.json", 
                results_snapshot=results_snapshot,
                round_idx=Round,
                trigger_round=LLM_TRIGGER_ROUND,
                allow_architecture_change=False, 
            )

            if new_config is not None:
                print("Applying LLM-updated hyperparameters")

                LEARNING_RATE = new_config.get("LEARNING_RATE", LEARNING_RATE)
                ROUNDS   = new_config.get("ROUNDS", ROUNDS)
                EPOCHS       = new_config.get("EPOCHS", EPOCHS) 

                print(
                    f"   ➜ LR={LEARNING_RATE}, "
                    f"ROUNDS={ROUNDS}, "
                    f"EPOCHS={EPOCHS}"
                )

                print("LLM update applied — continuing training\n")
    
    # Log total experiment time
    total_time = time.time() - experiment_start_time
    bench.log_experiment_time(total_time)

    if llm_trigger_time:
        bench.log_llm_overhead(
            total_time - (llm_trigger_time - experiment_start_time))

    print("Centralized Round-by-Round training completed.")


<font color='Brown'>***Federated Learning***</font>
---

In [26]:
# Containers for per-client and per-round tracking
prediction_dict= {}
actual_dict= {}
accuracy_dict= {}
loss_dict= {}

def architecture_changed(old_cfg, new_cfg):
    """
    Check whether model architecture-related hyperparameters have changed.
    """
    return (
        old_cfg["HIDDEN1_SIZE"] != new_cfg["HIDDEN1_SIZE"] or
        old_cfg["HIDDEN2_SIZE"] != new_cfg["HIDDEN2_SIZE"] or 
        old_cfg["DROUPOUT_RATE"] != new_cfg["DROUPOUT_RATE"]
    )


In [27]:
# Parameter serialization utilities for federated learning
if FL:
    def get_parameters(model) -> List[np.ndarray]:
        """
        Extract model parameters as NumPy arrays.
        """
        return [val.cpu().numpy() for _, val in model.state_dict().items()]

    def set_parameters(model, parameters: List[np.ndarray]):
        """
        Load NumPy parameter arrays into the model state.
        """
        params_dict = zip(model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
        model.load_state_dict(state_dict, strict=True)

In [28]:
# Flower client implementation for federated training
if FL:        
    class FlowerClient(fl.client.NumPyClient):
        """
        Flower NumPyClient wrapper that trains locally and reports updated parameters.
        """
        def __init__(self, cid, model, trainloader, FL_Update):
            self.cid = cid
            self.model = model
            self.trainloader = trainloader
            self.FL_Update = FL_Update

        def get_parameters(self, config):
            """
            Return the current local model parameters.
            """
            print(f"[Client {self.cid}] get_parameters")
            return get_parameters(self.model)


        def fit(self, parameters, config):
            """
            Run local training using server-provided parameters and return updated weights.
            """
            import traceback
            try:
                local_epochs = config.get("local_epochs", EPOCHS)
                lr = config.get("learning_rate", LEARNING_RATE)
                print(f"[Client {self.cid}, round {self.FL_Update}] fit, config: {config}")

               # Load server parameters into local model
                set_parameters(self.model, parameters)

               # Train locally and log per-client results
                _1, _2, _3, _4 = train(self.model, self.trainloader, epochs=local_epochs,lr=lr)
                prediction_dict[f'C{self.cid}R{self.FL_Update}'] = _1
                actual_dict[f'C{self.cid}R{self.FL_Update}'] = _2
                accuracy_dict[f'C{self.cid}R{self.FL_Update}'] = _3
                loss_dict[f'C{self.cid}R{self.FL_Update}'] = _4

               # Determine number of examples contributed by this client
                num_examples = 0
                try:
                    num_examples = len(self.trainloader.dataset)
                except Exception:
                    # Fallback: sum batch sizes (non-consuming) if dataset missing
                    try:
                        num_examples = sum(batch[0].shape[0] for batch in self.trainloader)
                    except Exception:
                        num_examples = 0

                return get_parameters(self.model), int(num_examples), {}

            except Exception as e:
                tb = traceback.format_exc()
                print(f"[Client {self.cid}] fit exception: {e}\n{tb}")

                # Return current parameters as fallback so the server can continue
                try:
                    return get_parameters(self.model), 0, {}
                except Exception as e2:
                    print(f"[Client {self.cid}] Failed to return fallback parameters: {e2}")
                    raise

        def evaluate(self, parameters, config):
            """
            No client-side evaluation (server handles evaluation separately).
            """
            print(f"[Client {self.cid}")
            print(f"[Client {self.cid}] evaluate, config: {config}")
            return None


In [29]:
# Utilities to rank client updates by parameter change magnitude
if FL:
    def retrieve_and_sort_client_updates(global_model, round_number, client_ids):
        """
        Load per-client update files for a round and rank clients by update magnitude.
        """
        client_updates = {}

        # Load saved client updates from disk
        for cid in client_ids:
            update_filename = f'EdgeCooperation/Performance/Results/C{cid}R{round_number}_update.pkl'
            with open(update_filename, 'rb') as update_file:
                client_update = pickle.load(update_file)
                client_updates[cid] = client_update

        # Compute contribution scores and sort clients       
        client_contributions = {
            cid: calculate_weight_magnitude(global_model, update)
            for cid, update in client_updates.items()
        }
        sorted_clients = sorted(client_contributions.items(), key=lambda x: x[1], reverse=True)

        # Return full ranking and the lowest contributors
        least_contributing_clients = sorted_clients[-3:]
        return sorted_clients, least_contributing_clients


    def calculate_weight_magnitude(global_model, client_update):
        """
        Compute the L2 norm of parameter differences between global and client-updated weights.

        Args:
            global_model (nn.Module): Global model before the client update.
            client_update (List[np.ndarray]): Client-updated parameters.

        Returns:
            float: L2 norm of parameter differences.
        """
        weight_diff = 0.0

        # Compare each parameter tensor against the corresponding client parameter array
        global_parameters = [param.detach().cpu().numpy() for param in global_model.parameters()]
        for global_param, client_param in zip(global_parameters, client_update):
            weight_diff += np.linalg.norm(global_param - client_param)
            
        return weight_diff


In [30]:
# Client factory for Flower simulation 
if FL:
    def General_Client():
        """
        Build and return a Flower client_fn for the given round.
        """
        def client_fn(cid: int, Round: int) -> FlowerClient:
            clients_ids_list = TrainingListPerRound[Round]

            # Only construct clients scheduled for this round
            if int(cid) in clients_ids_list:
                model = build_model().to(DEVICE)
                trainloader = Dataloaders[Round][int(cid)]
                arg_ = Round
                return FlowerClient(cid, model, trainloader, arg_)

            # Fail fast if an unexpected client id is requested
            raise ValueError(f"Client ID {cid} not found in the list for round {Round}")
            
        return client_fn

In [31]:
# Custom FedAvg strategy that saves the aggregated global model each round
if FL:
    Global_Models = {}

    class SaveModelStrategy(fl.server.strategy.FedAvg):
        """
        FedAvg strategy that persists the aggregated global model after each fit round.
        """

        def __init__(self, additional_argument, *args, **kwargs):
            """
            Args:
                additional_argument: Round index used for saving and bookkeeping.
            """
            super().__init__(*args, **kwargs)
            self.additional_argument = additional_argument


        def aggregate_fit(self, rnd: int, results: List[Tuple[fl.server.client_proxy.ClientProxy, fl.common.FitRes]], failures: List[BaseException]) -> Optional[fl.common.NDArrays]:
            """
            Aggregate client updates, load into a PyTorch model, and save a checkpoint for the round.
            """
            print(f"[SaveModelStrategy] aggregate_fit called for additional_argument={self.additional_argument}, rnd={rnd}, results={len(results)}, failures={len(failures)}")
            
            try:
                aggregated_parameters_tuple = super().aggregate_fit(rnd, results, failures)
                if not aggregated_parameters_tuple:
                    print(f"[SaveModelStrategy] No aggregated parameters returned for round {self.additional_argument}")
                    return aggregated_parameters_tuple

                aggregated_parameters, _ = aggregated_parameters_tuple
                if aggregated_parameters is None:
                    print(f"[SaveModelStrategy] aggregated_parameters is None for round {self.additional_argument}")
                    return aggregated_parameters_tuple

                # Convert Flower Parameters to numpy arrays
                aggregated_weights: List[np.ndarray] = fl.common.parameters_to_ndarrays(aggregated_parameters)

                # Load aggregated weights into a fresh model instance
                Global_Models[self.additional_argument] = build_model()
                params_dict = zip(Global_Models[self.additional_argument].state_dict().keys(), aggregated_weights)
                state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
                
                try:
                    Global_Models[self.additional_argument].load_state_dict(state_dict, strict=True)
                except Exception as e:
                    print(f"[SaveModelStrategy] Failed loading state_dict into model for round {self.additional_argument}: {e}")

                # Save round checkpoint 
                try:
                    os.makedirs(PATH, exist_ok=True)
                    save_path = f"{PATH}/GlobalModel_{self.additional_argument}.pth"
                    torch.save(Global_Models[self.additional_argument].state_dict(), save_path)
                    print(f"[SaveModelStrategy] Saved GlobalModel to {save_path}")
                except Exception as e:
                    print(f"[SaveModelStrategy] Failed to save GlobalModel_{self.additional_argument}: {e}")

                return aggregated_parameters_tuple

            except Exception as e:
                print(f"[SaveModelStrategy] Exception in aggregate_fit: {e}")
                raise
            


In [32]:
# Per-round configuration sent from server to clients
if FL:
    def fit_config(server_round: int):
        """
        Build the config dict passed to clients for each fit round.
        """
        return {
            "current_round": server_round,
            "local_epochs": CFG["EPOCHS"],
            "learning_rate": CFG["LEARNING_RATE"],
        }


In [33]:
def net2wider_load(target_model, source_state_dict, noise_std=1e-5):
    """
    Load weights from a source state_dict into a target model using Net2Wider-style expansion.

    Returns:
        copied_exact: Keys copied with exact shape match.
        copied_partial: Keys copied via widening/replication logic.
        skipped: Keys skipped due to missing keys or incompatible shapes.
    """
    tgt_state = target_model.state_dict()
    copied_exact, copied_partial, skipped = [], [], []

    for k, v in source_state_dict.items():
        if k not in tgt_state:
            skipped.append(k)
            continue
            
        tgt_v = tgt_state[k]
        
        # Case 1: Exact Match
        if tgt_v.shape == v.shape:
            tgt_state[k] = v.clone()
            copied_exact.append(k)
            continue

        # Case 2: Net2Wider Transformation (Partial/Expanded Copy)
        try:
            # Handle Linear Layer Weights (2D)
            if v.ndim == 2 and tgt_v.ndim == 2:
                o_new, i_new = tgt_v.shape
                o_old, i_old = v.shape
                
                # Create replication maps (Modulo logic)
                row_map = torch.arange(o_new) % o_old
                col_map = torch.arange(i_new) % i_old
                
                # Step 1: Replicate existing weights
                new_w = v[row_map][:, col_map]
                
                # Step 2: Scale outgoing weights (Function Preservation)
                # Count how many times each input index was replicated
                counts = torch.bincount(col_map, minlength=i_old).float()
                scaling = 1.0 / counts[col_map]
                new_w = new_w * scaling.view(1, -1)
                
                # Step 3: Symmetry Breaking (Tiny noise on new slots only)
                if noise_std > 0:
                    noise = torch.randn_like(new_w) * noise_std
                    mask = torch.ones_like(new_w)
                    mask[:o_old, :i_old] = 0 # Protect the original weights
                    new_w = new_w + (noise * mask)
                
                tgt_state[k] = new_w
                copied_partial.append(k)

            # Handle Biases or 1D Vectors
            elif v.ndim == 1 and tgt_v.ndim == 1:
                l_new, l_old = tgt_v.shape[0], v.shape[0]
                idx_map = torch.arange(l_new) % l_old
                tgt_state[k] = v[idx_map].clone()
                copied_partial.append(k)
            
            else:
                skipped.append(k)

        except Exception:
            skipped.append(k)

    # Apply the transformed state to the model
    target_model.load_state_dict(tgt_state)
    
    return copied_exact, copied_partial, skipped

In [34]:
# Federated learning training loop with optional mid-training LLM updates
importlib.reload(benchmark)
importlib.reload(llm)

if FL:
    import time
    os.makedirs(PATH, exist_ok=True)

   # Load initial global model checkpoint
    print("Loading Initial Global Model")
    Global_Models = {}
    Global_Models[0] = build_model()

    init_ckpt = f"0_Input_Random_Model_{MODE}.pth"
    if not os.path.exists(init_ckpt):
        raise FileNotFoundError(f"Missing initial checkpoint: {init_ckpt}")

    Global_Models[0].load_state_dict(torch.load(init_ckpt, map_location=DEVICE))
    Global_Models[0].train()

    # Define client participation per round
    TrainingListPerRound = {
        rnd: list(range(NUM_CLIENTS))
        for rnd in range(1, ROUNDS + 1)
    }

    # Initialize benchmark logger
    bench = BenchmarkLogger(
        output_path=PATH,
        mode="federated",
        model_type=MODE,
    )

    # Track experiment timing
    experiment_start_time = time.time()
    llm_trigger_time = None

    # Run FL one round at a time to allow dynamic config updates
    Round = 1
    while Round <= ROUNDS:

        bench.start_round(Round)

        print("\n" + "=" * 60)
        print(f"Starting FL Round {Round}")
        print("=" * 60)

        # Use previous round's global model as initialization
        prev_model = Global_Models.get(Round - 1, Global_Models[0])

        # Build strategy for saving aggregated global weights
        strategy = SaveModelStrategy(
            fraction_fit=1.0,
            fraction_evaluate=0.0,
            min_fit_clients=2,
            min_evaluate_clients=0,
            min_available_clients=2,
            on_fit_config_fn=fit_config,
            initial_parameters=fl.common.ndarrays_to_parameters(
                get_parameters(prev_model)
            ),
            additional_argument=Round,
        )

        client_factory = General_Client()

        # Run a single FL round
        t0 = time.time()
        fl.simulation.start_simulation(
            client_fn=lambda cid, rnd=Round: client_factory(cid, rnd),
            num_clients=NUM_CLIENTS,
            config=fl.server.ServerConfig(num_rounds=1),
            client_resources={"num_cpus": 16, "num_gpus": 1},
            ray_init_args={"num_cpus": 16, "num_gpus": 1},
            strategy=strategy,
        )
        agg_time = time.time() - t0

        # Track samples used this round
        samples_this_round = sum(
            len(Dataloaders[Round][cid].dataset)
            for cid in range(NUM_CLIENTS)
        )

        # Load aggregated global model from disk
        model_path = f"{PATH}/GlobalModel_{Round}.pth"
        Global_Models[Round] = build_model()
        Global_Models[Round].load_state_dict(
            torch.load(model_path, map_location=DEVICE)
        )
        Global_Models[Round].train()

        # Evaluate aggregated model on shared test set
        t0 = time.time()
        avg_loss, avg_acc, *_ = test(Global_Models[Round], Dataloaders["Test"])
        eval_time = time.time() - t0

        # Log round metrics
        bench.log_memory()
        bench.end_round(
            train_time=agg_time,
            eval_time=eval_time,
            accuracy=avg_acc,
            loss=avg_loss,
            samples_this_round=samples_this_round,
            learning_rate=CFG["LEARNING_RATE"],
            epochs=CFG["EPOCHS"],
        )
        
        # Trigger optional LLM mid-training update
        if Round == LLM_TRIGGER_ROUND and LLM==True:

            print("\nTriggering LLM mid-training update (FL)")
            llm_trigger_time = time.time()

            # Collect last local accuracy per client for this round
            client_accs = [
                accuracy_dict[k][-1]
                for k in accuracy_dict
                if k.endswith(f"R{Round}") and accuracy_dict[k]
            ]

            fl_results_snapshot = build_llm_fl_results_snapshot(
                global_acc=[avg_acc],
                client_accs=client_accs,
            )

            new_config = llm_mid_training_update(
                model_path=model_path,
                config_path="config.json",
                results_snapshot=fl_results_snapshot,
                round_idx=Round,
                trigger_round=LLM_TRIGGER_ROUND,
                allow_architecture_change=False,
            )

            # Apply updated config values
            if new_config:
                old_cfg = CFG.copy()
                CFG.update(new_config)

                LEARNING_RATE = CFG["LEARNING_RATE"]  
                EPOCHS = CFG["EPOCHS"]                
                ROUNDS = CFG["ROUNDS"]                

                print(
                    f"➜ Updated LR={LEARNING_RATE}, "
                    f"local_epochs={EPOCHS}, "
                    f"total_rounds={ROUNDS}"
                )

        print(f"End of FL Round {Round}")
        Round += 1

    # Log total experiment time
    total_time = time.time() - experiment_start_time
    bench.log_experiment_time(total_time)

    if llm_trigger_time:
        bench.log_llm_overhead(
            total_time - (llm_trigger_time - experiment_start_time)
        )

    print("\n==========================================")
    print(" FEDERATED TRAINING COMPLETED")
    print(f" Total runtime: {total_time:.2f} seconds")
    print("==========================================")

Loading Initial Global Model


INFO flwr 2026-02-09 21:41:11,505 | app.py:175 | Starting Flower simulation, config: ServerConfig(num_rounds=1, round_timeout=None)



Starting FL Round 1


2026-02-09 21:41:15,579	INFO worker.py:1771 -- Started a local Ray instance.
INFO flwr 2026-02-09 21:41:24,984 | app.py:210 | Flower VCE: Ray initialized with resources: {'accelerator_type:T4': 1.0, 'CPU': 16.0, 'memory': 6758731776.0, 'node:__internal_head__': 1.0, 'node:10.192.11.215': 1.0, 'object_store_memory': 3379365888.0, 'GPU': 1.0}
INFO flwr 2026-02-09 21:41:24,985 | app.py:224 | Flower VCE: Resources for each Virtual Client: {'num_cpus': 16, 'num_gpus': 1}
INFO flwr 2026-02-09 21:41:25,016 | app.py:270 | Flower VCE: Creating VirtualClientEngineActorPool with 1 actors
INFO flwr 2026-02-09 21:41:25,018 | server.py:89 | Initializing global parameters
INFO flwr 2026-02-09 21:41:25,022 | server.py:272 | Using initial parameters provided by strategy
INFO flwr 2026-02-09 21:41:25,022 | server.py:91 | Evaluating initial parameters
INFO flwr 2026-02-09 21:41:25,023 | server.py:104 | FL starting
DEBUG flwr 2026-02-09 21:41:25,024 | server.py:222 | fit_round 1: strategy sampled 48 clien

(DefaultActor pid=428104) [Client 26, round 1] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=428104) Epoch 1/3 | Loss: 2.522401 | Acc: 0.3854
(DefaultActor pid=428104) Epoch 2/3 | Loss: 2.133654 | Acc: 0.4948
(DefaultActor pid=428104) Epoch 3/3 | Loss: 2.054516 | Acc: 0.4948
(DefaultActor pid=428104) [Client 39, round 1] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=428104) Epoch 1/3 | Loss: 2.272067 | Acc: 0.6328
(DefaultActor pid=428104) Epoch 2/3 | Loss: 1.560113 | Acc: 0.7188
(DefaultActor pid=428104) Epoch 3/3 | Loss: 1.283851 | Acc: 0.7188
(DefaultActor pid=428104) [Client 44, round 1] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=428104) Epoch 1/3 | Loss: 2.383617 | Acc: 0.4557
(DefaultActor pid=428104) Epoch 2/3 | Loss: 1.586362 | Acc: 0.7656
(DefaultActor pid=428104) Epoch 3/3 | Loss: 1.242964 | Acc: 0.7656
(DefaultActor pid=4281

...

(DefaultActor pid=452276) [Client 40, round 8] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=452276) Epoch 1/3 | Loss: 1.227195 | Acc: 0.5104
(DefaultActor pid=452276) Epoch 2/3 | Loss: 1.136075 | Acc: 0.5833
(DefaultActor pid=452276) Epoch 3/3 | Loss: 1.064922 | Acc: 0.6328
(DefaultActor pid=452276) [Client 13, round 8] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=452276) Epoch 1/3 | Loss: 1.138983 | Acc: 0.5391
(DefaultActor pid=452276) Epoch 2/3 | Loss: 1.075655 | Acc: 0.5833
(DefaultActor pid=452276) Epoch 3/3 | Loss: 1.023882 | Acc: 0.5807
(DefaultActor pid=452276) [Client 2, round 8] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=452276) Epoch 1/3 | Loss: 1.259287 | Acc: 0.5417
(DefaultActor pid=452276) Epoch 2/3 | Loss: 1.185470 | Acc: 0.5521
(DefaultActor pid=452276) Epoch 3/3 | Loss: 1.123411 | Acc: 0.6042
(DefaultActor pid=45227

DEBUG flwr 2026-02-09 21:47:49,612 | server.py:236 | fit_round 1 received 48 results and 0 failures


(DefaultActor pid=452276) [Client 27, round 8] fit, config: {'current_round': 1, 'local_epochs': 3, 'learning_rate': 0.0025}
(DefaultActor pid=452276) Epoch 1/3 | Loss: 0.536356 | Acc: 0.7943
(DefaultActor pid=452276) Epoch 2/3 | Loss: 0.509582 | Acc: 0.8229
[SaveModelStrategy] aggregate_fit called for additional_argument=8, rnd=1, results=48, failures=0


WARNING flwr 2026-02-09 21:47:49,733 | fedavg.py:242 | No fit_metrics_aggregation_fn provided
INFO flwr 2026-02-09 21:47:49,746 | server.py:171 | evaluate_round 1: no clients selected, cancel
INFO flwr 2026-02-09 21:47:49,747 | server.py:153 | FL finished in 30.31462586500038
INFO flwr 2026-02-09 21:47:49,748 | app.py:225 | app_fit: losses_distributed []
INFO flwr 2026-02-09 21:47:49,748 | app.py:226 | app_fit: metrics_distributed_fit {}
INFO flwr 2026-02-09 21:47:49,749 | app.py:227 | app_fit: metrics_distributed {}
INFO flwr 2026-02-09 21:47:49,749 | app.py:228 | app_fit: losses_centralized []
INFO flwr 2026-02-09 21:47:49,750 | app.py:229 | app_fit: metrics_centralized {}


(DefaultActor pid=452276) Epoch 3/3 | Loss: 0.484578 | Acc: 0.8229
[SaveModelStrategy] Saved GlobalModel to TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_8.pth
Evaluation: loss=0.976076, acc=0.6004
End of FL Round 8

 FEDERATED TRAINING COMPLETED
 Total runtime: 402.50 seconds


<font color='Orange'>***Convert Pickle to JSON***</font>
---
---

In [35]:
def write_round_summary_json(
    run_path: str,
    round_num: int,
    output_name: str = None,
    cooccur_rate_threshold: float = 0.10,  # include predicted->predicted pairs >= 10% of that "from" label
    top_k_pairs: int = 25,
    top_k_labels: int = 10
):
    """
    Write a predictions-only JSON summary for a given round (no ground-truth required).
    """

    import os
    import json
    import pickle
    from collections import Counter

    # Label names must match the dataset's class encoding order
    LABEL_NAMES = [
        "Normal", "DDoS_UDP", "DDoS_ICMP", "DDoS_TCP", "DDoS_HTTP", "Password",
        "Vulnerability_scanner", "SQL_injection", "Uploading", "Backdoor", "Port_Scanning",
        "XSS", "Ransomware", "MITM", "OS_Fingerprinting"
    ]

    def load_pickle(filename):
        """
        Load a pickle file from the run directory.
        """
        full = os.path.join(run_path, filename)
        with open(full, "rb") as f:
            return pickle.load(f)

    # Load predictions only--
    pred = load_pickle(f"Global_{round_num}_pred")

    # Flatten batched predictions into a single sequence
    flat_pred = [int(x) for batch in pred for x in batch]

    def label_name(i: int) -> str:
        """
        Map a class index to a human-readable label.
        """
        return LABEL_NAMES[i] if 0 <= i < len(LABEL_NAMES) else f"class_{i}"

    n = len(flat_pred)
    if n == 0:
        raise ValueError(f"Round {round_num}: no samples found in predictions.")

    # Compute predicted label distribution
    pred_counts_raw = Counter(flat_pred)
    predicted_counts = {label_name(i): int(pred_counts_raw.get(i, 0)) for i in range(len(LABEL_NAMES))}
    predicted_percent = {k: (v / n) for k, v in predicted_counts.items()}
    inactive_labels = [k for k, v in predicted_counts.items() if v == 0]

    # Build predicted transition/co-occurrence pairs from consecutive predictions
    pair_counts_raw = Counter(zip(flat_pred[:-1], flat_pred[1:])) if n >= 2 else Counter()

    # Filter pairs by per-source-label rate threshold
    cooccur_pairs = []
    for (a, b), c in pair_counts_raw.items():
        if a == b:
            continue
        total_a = pred_counts_raw.get(a, 0)
        rate = (c / total_a) if total_a > 0 else 0.0
        if rate >= cooccur_rate_threshold:
            cooccur_pairs.append({
                "from_predicted": label_name(a),
                "to_predicted": label_name(b),
                "count": int(c),
                "rate": float(round(rate, 4))
            })

    cooccur_pairs.sort(key=lambda x: x["rate"], reverse=True)

    # Top-K lists for readability
    top_labels = [
        {"label": label_name(i), "count": int(cnt), "percent": float(round(cnt / n, 4))}
        for i, cnt in pred_counts_raw.most_common(top_k_labels)
    ]

    top_cooccur_by_count = [
        {"from_predicted": label_name(a), "to_predicted": label_name(b), "count": int(c)}
        for (a, b), c in pair_counts_raw.most_common(top_k_pairs)
        if a != b
    ]

    # Build summary payload
    summary = {
        "round": int(round_num),
        "num_samples": int(n),

        "predicted_counts": predicted_counts,
        "predicted_percent": {k: float(round(v, 6)) for k, v in predicted_percent.items()},
        "inactive_labels": inactive_labels,
        "top_labels": top_labels,
        "cooccur_pairs": cooccur_pairs,
        "top_cooccur_by_count": top_cooccur_by_count,
        "cooccur_rate_threshold": float(cooccur_rate_threshold)
    }

    # Choose default output filename    
    if output_name is None:
        output_name = f"Round_{round_num}_Summary.json"

    # Write JSON to disk
    out_path = os.path.join(run_path, output_name)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"[JSON] Saved {out_path}")

<font color='Grey'>***Performance Testing***</font>
---
---


In [36]:
# Locate saved global model checkpoints
directory = PATH + '/'
pattern = "GlobalModel_*.pth"

# Find all matching checkpoint files
files = glob.glob(os.path.join(directory, pattern))

# Extract round numbers from filenames
numbers = []
for file in files:
    base_name = os.path.basename(file)
    num_str = base_name.replace("GlobalModel_", "").replace(".pth", "")
    try:
        numbers.append(int(num_str))
    except ValueError:
        pass

# Determine the maximum round index
max_num = max(numbers) if numbers else 0
print(max_num)

# Iterate through all rounds up to the maximum
for num in range(1, max_num + 1):
    file_path = f"{PATH}/GlobalModel_{num}.pth"
    if os.path.exists(file_path):
        print(f"Loading {file_path}")
    else:
        print(f"File {file_path} does not exist")

8
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_1.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_2.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_3.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_4.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_5.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_6.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_7.pth
Loading TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/GlobalModel_8.pth


In [37]:
def view_pickle(path, G=None, suffix=None):
    """
    Inspect saved Global_* pickle files and print a small preview of their contents.
    """

    print(G)

    # Determine which rounds to inspect
    if G is not None:
        rounds = [G]
    else:
        # Automaically detect how many Global_X files exist
        rounds = sorted({
            int(f.split("_")[1])
            for f in os.listdir(path)
            if f.startswith("Global_") and f.split("_")[1].isdigit()
        })

    # Determine which suffixes to load  
    suffixes = [suffix] if suffix else ["pred", "actual", "accurracy", "loss"]

    # Load and preview each requested pickle
    for g in rounds:
        for s in suffixes:
            filename = os.path.join(path, f"Global_{g}_{s}")

            if not os.path.exists(filename):
                print(f"Missing file: {filename}")
                continue

            try:
                with open(filename, "rb") as f:
                    data = pickle.load(f)
                print(f"File: {filename}")

                if isinstance(data, list) and len(data) > 2:
                    pprint(data[:2])
                    print(f"... ({len(data)} total items)")
                else:
                    pprint(data)

            except Exception as e:
                print(f"❌ Error reading {filename}: {e}")

In [38]:
# Evaluate each saved global model on the test set and write per-round pickle + JSON outputs
pred_test = {}
actual_test = {}
accuracy_test = {}
loss_test = {}
G = 0

for num in range(1, max_num+1):
    # Load checkpoint for this round
    model = build_model()
    model.load_state_dict(torch.load(f"{PATH}/GlobalModel_{num}.pth"))
    model.eval()
    
    # Containers for this round’s evaluation outputs
    prediction_matrix = []
    actual_matrix= []
    acc_matrix = []
    loss_matrix=[]
    G = num

    # Run evaluation pass on the shared test loader
    criterion = nn.CrossEntropyLoss()
    correct, total, loss = 0, 0, 0.0

    with torch.no_grad():
        for images, labels in Dataloaders['Test']:
            outputs = model(images)

            loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            prediction_matrix.append(predicted.tolist())
            actual_matrix.append(labels.tolist())
    
    # Compute final metrics for this round
    loss /= len(Dataloaders['Test'].dataset)
    accuracy = correct / total

    loss_matrix.append(loss)
    acc_matrix.append(accuracy) 

    # Store results in dictionaries keyed by round
    pred_test[f'Global_{G}'] = prediction_matrix
    actual_test[f'Global_{G}'] = actual_matrix
    accuracy_test[f'Global_{G}'] = acc_matrix
    loss_test[f'Global_{G}'] = loss_matrix 

    filename = f"{PATH}/Global_{G}_pred"
    with open(filename, "wb") as outfile:
        pickle.dump(pred_test[f"Global_{G}"], outfile)

    filename = f"{PATH}/Global_{G}_actual"
    with open(filename, "wb") as outfile:
        pickle.dump(actual_test[f"Global_{G}"], outfile)

    filename = f"{PATH}/Global_{G}_accuracy"
    with open(filename, "wb") as outfile:
        pickle.dump(accuracy_test[f"Global_{G}"], outfile)

    filename = f"{PATH}/Global_{G}_loss"
    with open(filename, "wb") as outfile:
        pickle.dump(loss_test[f"Global_{G}"], outfile)

    # Write predictions-only JSON summary for this round
    write_round_summary_json(PATH, G)

[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_1_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_2_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_3_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_4_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_5_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_6_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_7_Summary.json
[JSON] Saved TT-FL-True-48-clients-14-atk-8-rounds-3-epochs-0.0025-lr-500-groups/Round_8_Summary.json
